# Model Calibration - Companion Notebook

This notebook contains only code and short operational notes that complement [`lecture_notes/09_model_calibration.pdf`](../../lecture_notes/09_model_calibration.pdf). The theory, formal definitions, and method-selection guidance are covered in the lecture notes.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize_scalar
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split

plt.style.use("seaborn-v0_8-whitegrid")
SEED = 42


This setup cell imports the libraries used throughout the notebook and fixes a random seed for reproducibility. It also defines the plotting style so that the figures have a consistent appearance across all experiments.


In [ ]:
def compute_ece(y_true, y_prob, n_bins=10, strategy="uniform"):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)

    if strategy == "uniform":
        bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    elif strategy == "quantile":
        bin_edges = np.quantile(y_prob, np.linspace(0.0, 1.0, n_bins + 1))
        bin_edges[0] = 0.0
        bin_edges[-1] = 1.0
    else:
        raise ValueError("strategy must be 'uniform' or 'quantile'")

    bin_ids = np.digitize(y_prob, bin_edges[:-1], right=False) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    rows = []
    ece = 0.0
    for bin_idx in range(n_bins):
        mask = bin_ids == bin_idx
        if not mask.any():
            continue

        conf = y_prob[mask].mean()
        acc = y_true[mask].mean()
        count = mask.sum()
        gap = abs(acc - conf)
        contrib = (count / n) * gap
        ece += contrib
        rows.append({
            "bin": bin_idx + 1,
            "conf(Bm)": conf,
            "acc(Bm)": acc,
            "|Bm|": count,
            "contribution": contrib,
        })

    return ece, pd.DataFrame(rows)


def plot_reliability_diagram(
    y_true,
    y_prob,
    *,
    n_bins=10,
    strategy="uniform",
    title="Calibration curve",
    ax=None,
    label="model",
):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))

    CalibrationDisplay.from_predictions(
        y_true,
        y_prob,
        n_bins=n_bins,
        strategy=strategy,
        name=label,
        ax=ax,
        color="tab:red",
    )
    ax.plot([0, 1], [0, 1], "k--", linewidth=1)
    ax.set_title(title)
    return ax


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


class TemperatureScaling:
    def __init__(self):
        self.temperature_ = 1.0

    def _nll(self, temperature, logits, y_true):
        prob = sigmoid(logits / temperature)
        prob = np.clip(prob, 1e-9, 1 - 1e-9)
        return -np.mean(y_true * np.log(prob) + (1 - y_true) * np.log(1 - prob))

    def fit(self, logits, y_true):
        result = minimize_scalar(
            self._nll,
            args=(np.asarray(logits), np.asarray(y_true)),
            bounds=(0.05, 10.0),
            method="bounded",
        )
        self.temperature_ = result.x
        return self

    def predict_proba(self, logits):
        return sigmoid(np.asarray(logits) / self.temperature_)


This cell defines the helper functions used repeatedly in the notebook.

- `compute_ece(...)` implements Expected Calibration Error manually and returns both the scalar value and a per-bin summary table.
- `plot_reliability_diagram(...)` wraps `CalibrationDisplay.from_predictions(...)` so that the same plotting logic can be reused across experiments.
- `TemperatureScaling` is a minimal post-hoc calibrator for models that expose logits through `decision_function`.

Keeping these pieces isolated makes the later sections shorter and easier to interpret.


## Worked Example

A direct code reproduction of the toy example from the lecture notes.


In [ ]:
y_prob_toy = np.array([0.10, 0.15, 0.30, 0.35, 0.55, 0.60, 0.70, 0.85, 0.90, 0.95])
y_true_toy = np.array([0, 0, 1, 1, 1, 1, 0, 1, 1, 1])

M = 5
bin_edges = np.linspace(0.0, 1.0, M + 1)
bin_ids = np.digitize(y_prob_toy, bin_edges[:-1], right=False) - 1
bin_ids = np.clip(bin_ids, 0, M - 1)

summary_toy = (
    pd.DataFrame({"bin": bin_ids + 1, "prob": y_prob_toy, "label": y_true_toy})
    .groupby("bin")
    .agg(
        **{
            "conf(Bm)": ("prob", "mean"),
            "acc(Bm)": ("label", "mean"),
            "|Bm|": ("label", "size"),
        }
    )
    .reset_index()
)

summary_toy


This cell constructs the bin-level table for the toy example. The output is useful because it makes the reliability diagram concrete: each plotted point corresponds to one row of this summary.

When reading the table, compare `conf(Bm)` and `acc(Bm)` directly. Their gap is the local calibration error for that bin, while `|Bm|` tells you how much weight that bin contributes to the overall metric.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.plot(summary_toy["conf(Bm)"], summary_toy["acc(Bm)"], marker="o", color="tab:red")

for _, row in summary_toy.iterrows():
    ax.annotate(
        f"B{int(row['bin'])}",
        (row["conf(Bm)"], row["acc(Bm)"]),
        textcoords="offset points",
        xytext=(5, 5),
    )

ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed fraction of positives")
ax.set_title("Toy example with manual binning")
ax.legend()
plt.show()


This figure is the visual counterpart of the table above. The diagonal is the perfect-calibration reference, and each labeled point shows where one bin falls relative to that ideal.

A practical way to read the plot is:
- points below the diagonal indicate overconfidence,
- points above the diagonal indicate underconfidence,
- points near the diagonal indicate better local calibration.

Because the bins are explicitly annotated, you can connect each geometric pattern in the plot back to a specific row in the table.


## Metrics

Manual ECE together with Brier score and log loss on the same toy example.


In [ ]:
ece_toy, ece_table_toy = compute_ece(y_true_toy, y_prob_toy, n_bins=5)

print(ece_table_toy.round(3).to_string(index=False))
print(f"\nECE (5 bins): {ece_toy:.4f}")
print(f"Brier score:  {brier_score_loss(y_true_toy, y_prob_toy):.4f}")
print(f"Log loss:     {log_loss(y_true_toy, y_prob_toy):.4f}")


This cell compares three complementary calibration-oriented summaries on exactly the same predictions.

- `ECE` emphasizes bin-level mismatch between confidence and observed frequency.
- `Brier score` measures squared probabilistic error at the instance level.
- `Log loss` penalizes incorrect high-confidence predictions more sharply.

The point is not that one metric replaces the others, but that they highlight different aspects of probabilistic quality. Reading them together gives a more complete diagnosis than any single scalar alone.


## End-to-End Pipeline

A runnable four-way split example: base model, calibration set, and before/after evaluation.


In [ ]:
X, y = make_classification(
    n_samples=5000,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    weights=[0.7, 0.3],
    class_sep=0.8,
    flip_y=0.08,
    random_state=SEED,
)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
X_train, X_rest, y_train, y_rest = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=SEED
)
X_val, X_cal, y_val, y_cal = train_test_split(
    X_rest, y_rest, test_size=0.50, stratify=y_rest, random_state=SEED
)

print(f"train={len(X_train)} | val={len(X_val)} | cal={len(X_cal)} | test={len(X_test)}")

base_model = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=SEED)
base_model.fit(X_train, y_train)

prob_base = base_model.predict_proba(X_test)[:, 1]


This cell sets up the full calibration workflow on synthetic data. The important implementation detail is the split structure: calibration is learned on `X_cal, y_cal`, not on the training data and not on the final test data.

Even though the example uses synthetic data for portability, the workflow mirrors what should happen on a real project: train the base model first, reserve a separate calibration set, and evaluate the final calibrated probabilities only on the untouched test set.


In [ ]:
cal_sigmoid = CalibratedClassifierCV(base_model, method="sigmoid", cv="prefit")
cal_sigmoid.fit(X_cal, y_cal)
prob_sigmoid = cal_sigmoid.predict_proba(X_test)[:, 1]

cal_isotonic = CalibratedClassifierCV(base_model, method="isotonic", cv="prefit")
cal_isotonic.fit(X_cal, y_cal)
prob_isotonic = cal_isotonic.predict_proba(X_test)[:, 1]

rows = []
for name, prob in {
    "base": prob_base,
    "sigmoid": prob_sigmoid,
    "isotonic": prob_isotonic,
}.items():
    ece, _ = compute_ece(y_test, prob, n_bins=10)
    rows.append({
        "model": name,
        "ece": ece,
        "brier": brier_score_loss(y_test, prob),
        "log_loss": log_loss(y_test, prob),
        "auc": roc_auc_score(y_test, prob),
    })

pd.DataFrame(rows).sort_values("ece").reset_index(drop=True)


This cell fits two post-hoc calibrators, Platt scaling (`sigmoid`) and isotonic regression, and compares them to the uncalibrated base model using the same test set.

The output table is best read row by row across metrics. A lower `ece`, `brier`, or `log_loss` suggests improvement in probability quality, while `auc` tells you whether ranking performance changed. This makes the trade-off explicit: calibration is mainly about improving the probabilities themselves, not necessarily maximizing discrimination.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, (name, prob) in zip(
    axes,
    [("Base", prob_base), ("Platt / Sigmoid", prob_sigmoid), ("Isotonic", prob_isotonic)],
):
    ece, _ = compute_ece(y_test, prob, n_bins=10)
    plot_reliability_diagram(
        y_test,
        prob,
        n_bins=10,
        title=f"{name}\nECE={ece:.3f}",
        ax=ax,
        label=name,
    )

plt.tight_layout()
plt.show()


These reliability diagrams provide the visual counterpart of the metric table above. Looking at the plots next to one another helps answer a more specific question than the scalar metrics alone: *how* did calibration change across the probability range?

Use the figure to check whether a lower ECE corresponds to a genuinely more diagonal curve or whether the improvement comes from only a few bins. This is especially useful when two methods have similar summary metrics but different local behaviors.


## Temperature Scaling

A minimal example for models that expose logits through `decision_function`.


In [ ]:
X_ts, y_ts = make_classification(
    n_samples=4000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    class_sep=0.7,
    flip_y=0.12,
    random_state=0,
)

X_train_ts, X_tmp_ts, y_train_ts, y_tmp_ts = train_test_split(
    X_ts, y_ts, test_size=0.40, stratify=y_ts, random_state=0
)
X_cal_ts, X_test_ts, y_cal_ts, y_test_ts = train_test_split(
    X_tmp_ts, y_tmp_ts, test_size=0.50, stratify=y_tmp_ts, random_state=0
)

lr = LogisticRegression(C=1e4, max_iter=2000, random_state=0)
lr.fit(X_train_ts, y_train_ts)

logits_cal = lr.decision_function(X_cal_ts)
logits_test = lr.decision_function(X_test_ts)
prob_uncal = sigmoid(logits_test)

ts = TemperatureScaling().fit(logits_cal, y_cal_ts)
prob_temp = ts.predict_proba(logits_test)

print(f"Optimal temperature: {ts.temperature_:.4f}")
print(f"AUC before: {roc_auc_score(y_test_ts, prob_uncal):.4f}")
print(f"AUC after:  {roc_auc_score(y_test_ts, prob_temp):.4f}")


This cell isolates the temperature-scaling case by using a model that exposes logits directly. The fitted scalar `temperature_` is chosen to minimize negative log-likelihood on the calibration set.

The printed AUC values are included for a specific reason: temperature scaling is a monotone transformation of the logits, so it should preserve ranking almost exactly while changing the confidence calibration of the probabilities.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for ax, title, prob in [
    (axes[0], "Before temperature scaling", prob_uncal),
    (axes[1], f"After temperature scaling (T={ts.temperature_:.2f})", prob_temp),
]:
    ece, _ = compute_ece(y_test_ts, prob, n_bins=10)
    plot_reliability_diagram(
        y_test_ts,
        prob,
        n_bins=10,
        title=f"{title}\nECE={ece:.3f}",
        ax=ax,
    )

plt.tight_layout()
plt.show()


This final figure shows whether the calibrated probabilities become more aligned with the diagonal after applying the learned temperature. Reading it together with the printed `temperature_` and AUC values gives a compact summary of the method:

- the scalar temperature tells you the direction and intensity of the correction,
- the reliability diagrams show the local before/after effect, and
- the near-constant AUC confirms that ranking was preserved while calibration changed.
